Project Outline: Comparing vLLM vs. llama.cpp for LLM Serving

Setup

Model: Qwen2.5-1.5B-Instruct
Tested two inference stacks: vLLM (GPU, FP16) vs. llama.cpp (CPU, quantized GGUF)
Hardware: T4 GPU (Colab)

Model size

FP16 (vLLM): ~3.09 GB
GGUF Q4_K_M (llama.cpp): 1.04 GB — roughly 3x smaller

Throughput / speed

vLLM, batched (20 prompts at once): 328.54 tokens/sec aggregate
vLLM, sequential (1 prompt at a time): latency ranged 1.88s–7.71s per request
llama.cpp, sequential (CPU): [your measured tok/s]

Key finding: throughput ≠ latency

Throughput = total work done per second across many concurrent requests
Latency = time for one request to complete
vLLM's batching is optimized for the former, not the latter — a single request isn't faster on vLLM, but many requests together are handled far more efficiently

When to use which

vLLM → high-throughput serving, many concurrent users, GPU available, aggregate speed matters most
llama.cpp → cheap/CPU-only deployment, edge devices, offline apps, single-user scenarios where small footprint and no GPU dependency matter more than raw speed

Broader takeaway

Benchmarks need to match how a system is actually used — a batched test measures something different from a sequential test, and neither is "wrong," they just answer different questions
Quantization (FP16 → 4-bit GGUF) trades some model precision for a large drop in size, making CPU deployment feasible

In [ ]:
!pip install vllm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 5.6 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of cuda-python to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 316.0/316.0 MB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 103.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.7/211.7 kB 23.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.3/18.3 MB 85.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.7/322.7 kB 24.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 85.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 782.6/782.6 kB 54.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 8

In [ ]:
print('Uninstalling existing torch and torchaudio...')
!pip uninstall torch torchaudio -y
print('Attempting to install compatible torch and torchaudio for PyTorch CUDA 13.0 from nightly builds...')
!pip install --pre torch torchaudio --index-url https://download.pytorch.org/whl/nightly/cu130
print('Reinstalling vllm...')
!pip install vllm

Uninstalling existing torch and torchaudio...
Attempting to install compatible torch and torchaudio for PyTorch CUDA 13.0 from nightly builds...
Looking in indexes: https://download.pytorch.org/whl/nightly/cu130
  Using cached https://download-r2.pytorch.org/whl/nightly/cu130/torch-2.15.0.dev20260909%2Bcu130-cp313-cp313-manylinux_2_28_x86_64.whl.metadata (37 kB)
  Using cached https://download-r2.pytorch.org/whl/nightly/cu130/torchaudio-2.11.0.dev20260910%2Bcu130-cp310-abi3-manylinux_2_28_x86_64.whl.metadata (7.5 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 503.9/503.9 MB 58.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.0/216.0 MB 76.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.4/135.4 MB 129.9 MB/s eta 0:00:00
  Using cached https://download-r2.pytorch.org/whl/nightly/triton-3.8.0%2Bgitc01b6774-cp313-cp313-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (990 bytes)
Using cached https://download-r2.pytorch.org/whl/nightly/cu130/torc

In [ ]:

from google.colab import userdata
import os

# Get the token from Colab secrets
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
# Or, if you need it as a variable:
# hf_token = userdata.get('HF_TOKEN')

In [ ]:
import os
import sys
from contextlib import redirect_stdout, redirect_stderr

os.environ['RAY_DEDUP_LOGS'] = '0'

from vllm import LLM, SamplingParams

# Temporarily redirect stdout/stderr to /dev/null to avoid io.UnsupportedOperation: fileno
with open(os.devnull, 'w') as devnull_file:
    with redirect_stdout(devnull_file), redirect_stderr(devnull_file):
        llm = LLM(model="Qwen/Qwen2.5-1.5B-Instruct", enforce_eager=True)

params = SamplingParams(temperature=0.7, max_tokens=200)
outputs = llm.generate(["Explain RAG in 2 sentences."], params)
print(outputs[0].outputs[0].text)

INFO 09-11 10:21:46 [api_utils.py:286] non-default args: {'disable_log_stats': True, 'enforce_eager': True, 'model': 'Qwen/Qwen2.5-1.5B-Instruct'}


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

INFO 09-11 10:22:05 [model.py:684] Resolved architecture: Qwen2ForCausalLM
WARNING 09-11 10:22:05 [model.py:2302] Your device 'Tesla T4' (with compute capability 7.5) doesn't support torch.bfloat16. Falling back to torch.float16 for compatibility.
WARNING 09-11 10:22:05 [model.py:2355] Casting torch.bfloat16 to torch.float16.
INFO 09-11 10:22:05 [model.py:2021] Using max model len 32768
INFO 09-11 10:22:05 [scheduler.py:277] Chunked prefill is enabled with max_num_batched_tokens=8192.
WARNING 09-11 10:22:06 [vllm.py:1371] Enforce eager set, disabling torch.compile and CUDAGraphs. This is equivalent to setting -cc.mode=none -cc.cudagraph_mode=none
WARNING 09-11 10:22:06 [vllm.py:1406] Inductor compilation was disabled by user settings, optimizations settings that are only active during inductor compilation will be ignored.
INFO 09-11 10:22:06 [kernel.py:369] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['vllm_c', 'native'], fused_add_rms_norm=['vllm_

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

...

Rendering prompts:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

WARNING 09-11 10:24:07 [jit_monitor.py:141] Triton kernel JIT compilation during inference: kernel_unified_attention. This causes a latency spike; consider extending warmup to cover this shape/config.


Processed prompts: 100%|██████████| 1/1 [00:05<00:00,  5.97s/it, est. speed input: 1.51 toks/s, output: 20.12 toks/s]

 RAG is a term that refers to the ability of a machine learning model to accurately predict events or outcomes based on past data and the relationships between them. It is a powerful tool in the field of machine learning that has been used to make predictions in a variety of domains, including finance, healthcare, and marketing. RAG can help organizations make better decisions by enabling them to analyze large amounts of data and identify patterns and trends that might not be immediately apparent. It can also help organizations to make predictions and forecasts based on historical data, which can be used to inform decision-making and improve outcomes.


In [ ]:
import time

prompts = [f"Explain RAG in 2 sentences. (variant {i})" for i in range(20)]
params = SamplingParams(temperature=0.7, max_tokens=200)

start = time.time()
outputs = llm.generate(prompts, params)
elapsed = time.time() - start

total_output_tokens = sum(len(o.outputs[0].token_ids) for o in outputs)
total_input_tokens = sum(len(o.prompt_token_ids) for o in outputs)

print(f"Prompts: {len(prompts)}")
print(f"Elapsed: {elapsed:.2f}s")
print(f"Output tokens: {total_output_tokens}")
print(f"Throughput: {total_output_tokens / elapsed:.2f} tokens/sec (output)")
print(f"Requests/sec: {len(prompts) / elapsed:.2f}")
print(f"Avg output tokens/request: {total_output_tokens / len(prompts):.1f}")

Rendering prompts:   0%|          | 0/20 [00:00<?, ?it/s]

Processed prompts: 100%|██████████| 20/20 [00:09<00:00,  2.00it/s, est. speed input: 29.07 toks/s, output: 339.76 toks/s]

Prompts: 20
Elapsed: 10.03s
Output tokens: 3390
Throughput: 338.08 tokens/sec (output)
Requests/sec: 1.99
Avg output tokens/request: 169.5


In [ ]:
import time

prompts = [f"Explain RAG in 2 sentences. (variant {i})" for i in range(20)]

latencies = []
for p in prompts:
    start = time.time()
    llm.generate([p], params)
    latencies.append(time.time() - start)

print(f"Avg latency: {sum(latencies)/len(latencies):.2f}s")
print(f"Min/Max latency: {min(latencies):.2f}s / {max(latencies):.2f}s")

Rendering prompts:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts: 100%|██████████| 1/1 [00:03<00:00,  3.22s/it, est. speed input: 4.35 toks/s, output: 28.90 toks/s]


Rendering prompts:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts: 100%|██████████| 1/1 [00:07<00:00,  7.60s/it, est. speed input: 1.84 toks/s, output: 26.32 toks/s]


Rendering prompts:   0%|          | 0/1 [00:00<?, ?it/s]

...

Rendering prompts:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts: 100%|██████████| 1/1 [00:02<00:00,  2.94s/it, est. speed input: 5.10 toks/s, output: 29.61 toks/s]


Rendering prompts:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts: 100%|██████████| 1/1 [00:07<00:00,  7.54s/it, est. speed input: 1.99 toks/s, output: 26.54 toks/s]

Avg latency: 6.46s
Min/Max latency: 1.88s / 7.71s


In [ ]:
from huggingface_hub import list_repo_files
print(list_repo_files("Qwen/Qwen2.5-1.5B-Instruct-GGUF"))

['.gitattributes', 'LICENSE', 'README.md', 'qwen2.5-1.5b-instruct-fp16.gguf', 'qwen2.5-1.5b-instruct-q2_k.gguf', 'qwen2.5-1.5b-instruct-q3_k_m.gguf', 'qwen2.5-1.5b-instruct-q4_0.gguf', 'qwen2.5-1.5b-instruct-q4_k_m.gguf', 'qwen2.5-1.5b-instruct-q5_0.gguf', 'qwen2.5-1.5b-instruct-q5_k_m.gguf', 'qwen2.5-1.5b-instruct-q6_k.gguf', 'qwen2.5-1.5b-instruct-q8_0.gguf']


In [ ]:
from huggingface_hub import hf_hub_download

model_path = hf_hub_download(
    repo_id="Qwen/Qwen2.5-1.5B-Instruct-GGUF",
    filename="qwen2.5-1.5b-instruct-q4_k_m.gguf"
)
print(model_path)

qwen2.5-1.5b-instruct-q4_k_m.gguf: reconstructing file:   0%|          |  0.00B / 1.12GB            

qwen2.5-1.5b-instruct-q4_k_m.gguf: downloading bytes:           |  0.00B            

/root/.cache/huggingface/hub/models--Qwen--Qwen2.5-1.5B-Instruct-GGUF/snapshots/91cad51170dc346986eccefdc2dd33a9da36ead9/qwen2.5-1.5b-instruct-q4_k_m.gguf


In [ ]:
!pip install -q llama-cpp-python

from llama_cpp import Llama

llm_cpp = Llama(
    model_path=model_path,
    n_ctx=2048,
    n_threads=8,      # Colab CPU has multiple cores — use them
    n_gpu_layers=0,   # 0 = pure CPU; raise this later if you want GPU-accelerated llama.cpp
    verbose=False,
)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.9/74.9 MB 10.7 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 4.7 MB/s eta 0:00:00


In [ ]:
import os

# FP16 (used by vLLM) — find the cached snapshot folder
!du -sh ~/.cache/huggingface/hub/models--Qwen--Qwen2.5-1.5B-Instruct

# GGUF Q4_K_M (used by llama.cpp) — the single file you downloaded
gguf_size_gb = os.path.getsize(model_path) / (1024**3)
print(f"GGUF Q4_K_M size: {gguf_size_gb:.2f} GB")

du: cannot access '/root/.cache/huggingface/hub/models--Qwen--Qwen2.5-1.5B-Instruct': No such file or directory
GGUF Q4_K_M size: 1.04 GB


In [ ]:
from huggingface_hub import scan_cache_dir

cache_info = scan_cache_dir()
for repo in cache_info.repos:
    if "Qwen2.5-1.5B-Instruct" in repo.repo_id:
        print(f"{repo.repo_id}: {repo.size_on_disk / (1024**3):.2f} GB")

Qwen/Qwen2.5-1.5B-Instruct-GGUF: 1.04 GB


**Metric**	vLLM (FP16)	llama.cpp (GGUF Q4_K_M)
**Model size**	~3.1 GB	~0.9–1.1 GB
**Tokens/sec**	328.54 tok/s (batched, GPU)	[your result] tok/s (sequential, CPU)
**Hardware needed	GPU **(T4 here)	CPU-only works fine


llama.cpp is the pick for cheap, CPU-only, single-user or edge deployment (laptops, low-cost VMs, offline/on-device apps) where small footprint and zero GPU dependency matter most; vLLM is the pick for high-throughput serving with many concurrent users, where GPU batching lets you amortize compute across requests and maximize aggregate tokens/sec.

In [1]:
   from transformers import AutoModelForCausalLM, AutoTokenizer
   import torch
   model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-1.5B-Instruct", torch_dtype=torch.float16, device_map="cuda")

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [2]:
print(f"FP16 memory allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

FP16 memory allocated: 3.09 GB


Measuring KV cache memory growth

In [4]:
# Reload FP16 model if needed (or reuse model_4bit — either works for this demo)
import torch
def measure_kv_cache_memory(model, tokenizer, prompt, max_new_tokens):
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    baseline = torch.cuda.memory_allocated() / 1e9

    with torch.no_grad():
        model.generate(**inputs, max_new_tokens=max_new_tokens, use_cache=True)

    peak = torch.cuda.max_memory_allocated() / 1e9
    return baseline, peak, peak - baseline

In [5]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-1.5B-Instruct")

prompt = "Explain the history of the Roman Empire in detail."

for n_tokens in [50, 200, 500, 1000]:
    baseline, peak, cache_cost = measure_kv_cache_memory(model, tokenizer, prompt, n_tokens)
    print(f"max_new_tokens={n_tokens}: baseline={baseline:.2f}GB, peak={peak:.2f}GB, kv_cache≈{cache_cost:.3f}GB")

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

max_new_tokens=50: baseline=3.09GB, peak=3.11GB, kv_cache≈0.019GB
max_new_tokens=200: baseline=3.10GB, peak=3.11GB, kv_cache≈0.014GB
max_new_tokens=500: baseline=3.10GB, peak=3.12GB, kv_cache≈0.027GB
max_new_tokens=1000: baseline=3.10GB, peak=3.13GB, kv_cache≈0.037GB


In [6]:
def measure_batch_kv_cache(model, tokenizer, prompt, batch_size, max_new_tokens=200):
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

    prompts = [prompt] * batch_size
    inputs = tokenizer(prompts, return_tensors="pt", padding=True).to("cuda")

    with torch.no_grad():
        model.generate(**inputs, max_new_tokens=max_new_tokens, use_cache=True)

    return torch.cuda.max_memory_allocated() / 1e9

for bsz in [1, 4, 8, 16]:
    peak = measure_batch_kv_cache(model, tokenizer, prompt, bsz)
    print(f"batch_size={bsz}: peak memory={peak:.2f}GB")

batch_size=1: peak memory=3.11GB
batch_size=4: peak memory=3.16GB
batch_size=8: peak memory=3.22GB
batch_size=16: peak memory=3.35GB


The bigger picture takeaway

Two things are eating GPU memory on a very different scale:

Model weights (3.1 GB) — fixed, doesn't change no matter what you generate. This is the "cost of admission" just to load the model at all.
KV-cache (~0.02–0.04 GB in your test) — small here because your model is tiny (1.5B params) and you only generated up to 1000 tokens. But this is the number that becomes dangerous at scale: with bigger models (7B, 70B) and longer contexts (8K, 32K, 128K tokens) or many concurrent users, KV-cache can easily balloon to gigabytes or tens of gigabytes — sometimes bigger than the model weights themselves.

That's exactly why vLLM's core innovation (PagedAttention) exists: at large scale, KV-cache memory management becomes the actual bottleneck for how many users you can serve at once, not the model weights. Your tiny 1.5B test barely shows this effect — but the same experiment on a 70B model with long contexts would show KV-cache dominating memory use entirely.

Full project summary

Throughput vs. latency (vLLM): batched serving (20 prompts at once) hit 328.54 tok/s aggregate; sequential single-request calls took 1.88s–7.71s each — proving these are different measurements answering different questions, not two versions of the same number.

Serving stack comparison: vLLM (FP16, ~3.09 GB) targets high-throughput concurrent serving on GPU; llama.cpp (GGUF Q4_K_M, 1.04 GB) targets cheap, CPU-only, single-user or edge deployment — a ~3x size reduction traded for lower per-request GPU-batched throughput.

Quantization memory impact (Transformers/bitsandbytes): loading the same model in FP16 vs. 4-bit directly showed the raw memory cost of precision — 4-bit cut GPU memory roughly to a quarter of FP16, independent of which inference engine is used.

KV-cache growth: memory scales with both sequence length and batch size, not just model weight size — the cache can dominate total memory use at longer contexts or larger batches, which is the core problem PagedAttention (used in vLLM) was built to solve efficiently.

Big picture: serving an LLM well means managing three separate memory/speed levers — model precision (quantization), request batching, and KV-cache — and different tools (vLLM, llama.cpp, raw Transformers) make different tradeoffs across these three, suited to different deployment scenarios (high-concurrency GPU serving vs. lightweight CPU/edge use).